# Heterogeneous Graph Auto-Encoder — Credit Card Fraud Detection

Self-contained Colab notebook implementing *Majumder et al., Heterogeneous Graph
Auto-Encoder for Credit Card Fraud Detection* (IJCA 32(2), 2025).

**Idea:** model transactions as a heterogeneous graph (customer &harr; transaction &harr; merchant),
encode with a custom **intra-edge attention** GNN, reconstruct each node's attributes,
and train **only on genuine transactions** so that a high reconstruction error flags fraud
(anomaly detection &rarr; no resampling needed for class imbalance).

This notebook runs the whole pipeline top-to-bottom:
1. Setup &amp; data &nbsp; 2. Data processing (features + graph) &nbsp; 3. EDA charts
4. Model (custom attention) &nbsp; 5. Training &nbsp; 6. Validation &amp; thresholds &nbsp; 7. Result charts
8. Score a single transaction

> **Runtime:** enable a GPU in Colab (*Runtime &rarr; Change runtime type &rarr; GPU*) for a faster run.

## 1. Setup — install dependencies

In [ ]:
# Colab ships torch; add PyTorch Geometric.
%pip -q install torch_geometric
import torch, torch_geometric
print("torch", torch.__version__, "| pyg", torch_geometric.__version__,
      "| CUDA:", torch.cuda.is_available())

## 2. Get the dataset

Kaggle **kartik2112/fraud-detection** (`fraudTrain.csv`, `fraudTest.csv`).

**Option A - Kaggle API (recommended):** upload your `kaggle.json` when prompted, then this
cell downloads and unzips automatically.
**Option B - manual:** just upload `fraudTrain.csv` and `fraudTest.csv` when prompted.

In [ ]:
import os
from pathlib import Path

TRAIN_CSV, TEST_CSV = "fraudTrain.csv", "fraudTest.csv"

def _have_data():
    return Path(TRAIN_CSV).exists() and Path(TEST_CSV).exists()

if not _have_data():
    try:
        # Option A: Kaggle API
        from google.colab import files
        print("Upload kaggle.json (Kaggle > Account > Create New API Token), or cancel to upload CSVs manually.")
        up = files.upload()
        if "kaggle.json" in up:
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
                f.write(up["kaggle.json"])
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
            %pip -q install kaggle
            !kaggle datasets download -d kartik2112/fraud-detection -q
            !unzip -o -q fraud-detection.zip
    except Exception as e:
        print("Kaggle path skipped:", e)

if not _have_data():
    # Option B: manual upload of the two CSVs
    from google.colab import files
    print("Upload fraudTrain.csv and fraudTest.csv")
    files.upload()

assert _have_data(), "fraudTrain.csv / fraudTest.csv not found."
print("Dataset ready:", TRAIN_CSV, TEST_CSV)

## 3. Configuration

In [ ]:
# Hyper-parameters (paper Table 3, with the two forced deviations: depth and stable reparam).
CFG = dict(
    hidden_dim=64, heads=16, encoder_layers=2,   # paper says 124 -> oversmooths; 2 is runnable
    latent_dim=64, decoder_hidden=64, dropout=0.3,
    lr=2e-3, weight_decay=1e-4, epochs=40, val_fraction=0.15,
    beta_kl=0.0,          # 0 = faithful (no KL); try 1e-3 to add KL regularization (usually helps)
    train_size=100_000,   # rows drawn from the real train file (all fraud kept); None = full
    test_size=50_000,     # rows drawn from the real test file; None = full
    seed=42,
)
CFG

## 4. Data processing — features &amp; heterogeneous graph

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

# ---- consistent, colorblind-safe palette (Okabe-Ito): genuine=blue, fraud=orange ----
GENUINE, FRAUD = "#0072B2", "#D55E00"
ACCENT = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
THRC = "#333333"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.color": "#dddddd", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
    "axes.edgecolor": "#888888", "figure.dpi": 110,
})
print("device:", DEVICE)

In [ ]:
# ---- per-type feature definitions (each node type has its OWN attribute set) ----
CUST_CONT = ["age", "log_city_pop", "home_lat", "home_long"]; CUST_CAT = ["gender"]
MERC_CONT = ["merch_lat_f", "merch_long_f"];                  MERC_CAT = ["category"]
TXN_CONT  = ["log_amt", "log_distance", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]; TXN_CAT = []

def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180.0
    a = (0.5 - np.cos((lat2-lat1)*p)/2 + np.cos(lat1*p)*np.cos(lat2*p)*(1-np.cos((lon2-lon1)*p))/2)
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

class TypeFeaturizer:
    # Fit on genuine rows; transform any rows. Output = standardized continuous FIRST, then one-hot blocks.
    def __init__(self, kind, cont_cols, cat_cols):
        self.kind, self.cont_cols, self.cat_cols = kind, cont_cols, cat_cols
        self.cat_vocab, self.mean_, self.scale_ = {}, None, None

    def _raw(self, df):
        out = pd.DataFrame(index=df.index)
        if self.kind == "customer":
            ts = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
            dob = pd.to_datetime(df["dob"], errors="coerce")
            age = (ts.dt.year - dob.dt.year)
            out["age"] = age.fillna(age.median() if age.notna().any() else 40).astype(float)
            out["log_city_pop"] = np.log1p(pd.to_numeric(df["city_pop"], errors="coerce").fillna(0).clip(lower=0))
            out["home_lat"] = pd.to_numeric(df["lat"], errors="coerce").fillna(0.0)
            out["home_long"] = pd.to_numeric(df["long"], errors="coerce").fillna(0.0)
            out["gender"] = df["gender"].astype(str).str.upper().values
        elif self.kind == "merchant":
            out["merch_lat_f"] = pd.to_numeric(df["merch_lat"], errors="coerce").fillna(0.0)
            out["merch_long_f"] = pd.to_numeric(df["merch_long"], errors="coerce").fillna(0.0)
            out["category"] = df["category"].astype(str).values
        else:  # transaction
            out["log_amt"] = np.log1p(pd.to_numeric(df["amt"], errors="coerce").fillna(0).clip(lower=0))
            ts = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
            hour = ts.dt.hour.fillna(12).astype(float); dow = ts.dt.dayofweek.fillna(0).astype(float)
            out["hour_sin"] = np.sin(2*np.pi*hour/24); out["hour_cos"] = np.cos(2*np.pi*hour/24)
            out["dow_sin"] = np.sin(2*np.pi*dow/7);  out["dow_cos"] = np.cos(2*np.pi*dow/7)
            dist = haversine_km(pd.to_numeric(df["lat"], errors="coerce").fillna(0).values,
                                pd.to_numeric(df["long"], errors="coerce").fillna(0).values,
                                pd.to_numeric(df["merch_lat"], errors="coerce").fillna(0).values,
                                pd.to_numeric(df["merch_long"], errors="coerce").fillna(0).values)
            out["log_distance"] = np.log1p(np.clip(dist, 0, None))
        return out

    def fit(self, df):
        raw = self._raw(df)
        cont = raw[self.cont_cols].to_numpy(float)
        self.mean_ = cont.mean(0); std = cont.std(0); std[std < 1e-8] = 1.0; self.scale_ = std
        for c in self.cat_cols:
            self.cat_vocab[c] = sorted(raw[c].astype(str).unique().tolist())
        return self

    def transform(self, df):
        raw = self._raw(df)
        blocks = [(raw[self.cont_cols].to_numpy(float) - self.mean_) / self.scale_]
        for c in self.cat_cols:
            idx = {v: i for i, v in enumerate(self.cat_vocab[c])}
            oh = np.zeros((len(raw), len(self.cat_vocab[c])))
            for r, v in enumerate(raw[c].astype(str).values):
                if v in idx: oh[r, idx[v]] = 1.0
            blocks.append(oh)
        return np.hstack(blocks).astype(np.float32)

    @property
    def dim(self): return len(self.cont_cols) + sum(len(self.cat_vocab[c]) for c in self.cat_cols)
    @property
    def cat_groups(self):
        g, start = [], len(self.cont_cols)
        for c in self.cat_cols:
            w = len(self.cat_vocab[c]); g.append((c, start, w)); start += w
        return g

def make_featurizers():
    return {"customer": TypeFeaturizer("customer", CUST_CONT, CUST_CAT),
            "merchant": TypeFeaturizer("merchant", MERC_CONT, MERC_CAT),
            "transaction": TypeFeaturizer("transaction", TXN_CONT, TXN_CAT)}

def type_targets(feats):
    return {t: {"cont": len(f.cont_cols), "cat_groups": f.cat_groups} for t, f in feats.items()}

METADATA = (["customer", "merchant", "transaction"],
            [("customer","makes","transaction"), ("transaction","rev_makes","customer"),
             ("merchant","sells","transaction"), ("transaction","rev_sells","merchant")])

In [ ]:
# ---- load + subsample (keep all fraud), and build a HeteroData graph ----
from torch_geometric.data import HeteroData
REQ = ["trans_date_trans_time","cc_num","merchant","category","amt","gender",
       "city_pop","lat","long","merch_lat","merch_long","dob","is_fraud","trans_num"]

def load_raw(path, subsample, seed):
    df = pd.read_csv(path)
    if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
        df = df.drop(columns=df.columns[0])
    if subsample is not None and subsample < len(df):
        rng = np.random.default_rng(seed)
        fraud = df[df.is_fraud == 1]; gen = df[df.is_fraud == 0]
        n_gen = max(0, subsample - len(fraud))
        gi = rng.choice(gen.index.values, size=min(n_gen, len(gen)), replace=False)
        df = pd.concat([fraud, gen.loc[gi]]).sort_index()
    return df.reset_index(drop=True)

def build_graph(df, feats):
    cu, ci = np.unique(df["cc_num"].astype(str).values, return_inverse=True)
    mu, mi = np.unique(df["merchant"].astype(str).values, return_inverse=True)
    tx = np.arange(len(df))
    d = df.copy(); d["cc_num"] = d["cc_num"].astype(str); d["merchant"] = d["merchant"].astype(str)
    cdf = d.drop_duplicates("cc_num").set_index("cc_num").loc[cu].reset_index()
    mdf = d.drop_duplicates("merchant").set_index("merchant").loc[mu].reset_index()
    data = HeteroData()
    data["customer"].x = torch.from_numpy(feats["customer"].transform(cdf))
    data["merchant"].x = torch.from_numpy(feats["merchant"].transform(mdf))
    data["transaction"].x = torch.from_numpy(feats["transaction"].transform(df))
    data["transaction"].y = torch.tensor(df["is_fraud"].to_numpy(np.int64))
    c2t = np.vstack([ci, tx]).astype(np.int64); m2t = np.vstack([mi, tx]).astype(np.int64)
    data["customer","makes","transaction"].edge_index = torch.from_numpy(c2t)
    data["transaction","rev_makes","customer"].edge_index = torch.from_numpy(c2t[[1,0]])
    data["merchant","sells","transaction"].edge_index = torch.from_numpy(m2t)
    data["transaction","rev_sells","merchant"].edge_index = torch.from_numpy(m2t[[1,0]])
    return data

train_raw = load_raw(TRAIN_CSV, CFG["train_size"], CFG["seed"])
test_raw  = load_raw(TEST_CSV,  CFG["test_size"],  CFG["seed"])
print(f"train rows={len(train_raw)} fraud={int(train_raw.is_fraud.sum())} | "
      f"test rows={len(test_raw)} fraud={int(test_raw.is_fraud.sum())}")

# genuine-only training; validation = held-out genuine + all train fraud
rng = np.random.default_rng(CFG["seed"])
gen = train_raw[train_raw.is_fraud == 0]; fr = train_raw[train_raw.is_fraud == 1]
perm = rng.permutation(len(gen)); nval = int(len(gen)*CFG["val_fraction"])
train_gen = gen.iloc[perm[nval:]].reset_index(drop=True)
val_df = pd.concat([gen.iloc[perm[:nval]], fr]).sample(frac=1.0, random_state=CFG["seed"]).reset_index(drop=True)

feats = make_featurizers()
for k in feats: feats[k].fit(train_gen)          # fit on genuine training rows only
TT = type_targets(feats); IN_DIMS = {t: feats[t].dim for t in feats}
train_data = build_graph(train_gen, feats).to(DEVICE)
val_data   = build_graph(val_df, feats)
test_data  = build_graph(test_raw, feats)
val_labels = val_df["is_fraud"].to_numpy(); test_labels = test_raw["is_fraud"].to_numpy()
print("feature dims:", IN_DIMS)

## 5. EDA — data characteristics

In [ ]:
# Class balance, amount distribution, fraud rate by hour and by category.
def _counts(df):
    f = int((df.is_fraud==1).sum()); return len(df)-f, f
tg, tf = _counts(train_raw); eg, ef = _counts(test_raw)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# (a) class balance
x = np.arange(2); w = 0.38
ax[0,0].bar(x-w/2, [tg, eg], w, label="genuine", color=GENUINE)
ax[0,0].bar(x+w/2, [tf, ef], w, label="fraud", color=FRAUD)
ax[0,0].set_xticks(x); ax[0,0].set_xticklabels(["train","test"])
ax[0,0].set_title("Class balance"); ax[0,0].set_ylabel("rows"); ax[0,0].legend(frameon=False)
ax[0,0].set_yscale("log")

# (b) amount distribution (log1p) genuine vs fraud
la = np.log1p(pd.to_numeric(train_raw["amt"], errors="coerce").fillna(0).clip(lower=0).values)
y = train_raw.is_fraud.values
bins = np.linspace(la.min(), np.percentile(la, 99.5), 40)
ax[0,1].hist(la[y==0], bins=bins, color=GENUINE, alpha=0.65, label="genuine", density=True)
ax[0,1].hist(la[y==1], bins=bins, color=FRAUD, alpha=0.65, label="fraud", density=True)
ax[0,1].set_title("Transaction amount  (log 1+amt)"); ax[0,1].set_xlabel("log_amt")
ax[0,1].set_ylabel("density"); ax[0,1].legend(frameon=False)

# (c) fraud rate by hour
hour = pd.to_datetime(train_raw["trans_date_trans_time"], errors="coerce").dt.hour.fillna(12).astype(int)
byh = pd.DataFrame({"h": hour, "f": y}).groupby("h")["f"].mean().reindex(range(24), fill_value=0)
ax[1,0].bar(range(24), byh.values*100, color=FRAUD)
ax[1,0].set_title("Fraud rate by hour of day"); ax[1,0].set_xlabel("hour"); ax[1,0].set_ylabel("fraud %")

# (d) fraud rate by top merchant categories
cat = train_raw["category"].astype(str)
top = cat.value_counts().head(10).index.tolist()
byc = pd.DataFrame({"c": cat, "f": y})
rate = byc[byc.c.isin(top)].groupby("c")["f"].mean().reindex(top)*100
ax[1,1].barh(range(len(top)), rate.values, color=FRAUD)
ax[1,1].set_yticks(range(len(top))); ax[1,1].set_yticklabels(top, fontsize=8)
ax[1,1].invert_yaxis(); ax[1,1].set_title("Fraud rate by category (top 10 by volume)"); ax[1,1].set_xlabel("fraud %")

plt.tight_layout(); plt.show()

## 6. Model — HGAE with intra-edge attention

The distinctive layer: attention softmax is taken **across the H heads of each edge**
(not across neighbours), and neighbour messages are **summed uniformly** (paper Eq. 1-5).

In [ ]:
def _rk(rel): return "__".join(rel)

class HGAEConv(nn.Module):
    def __init__(self, metadata, hidden, heads):
        super().__init__()
        self.ntypes, self.etypes = metadata[0], metadata[1]
        self.H, self.dk = heads, hidden // heads
        self.s = nn.ModuleDict({t: nn.Linear(hidden, hidden) for t in self.ntypes})
        self.d = nn.ModuleDict({t: nn.Linear(hidden, hidden) for t in self.ntypes})
        self.m = nn.ModuleDict({t: nn.Linear(hidden, hidden) for t in self.ntypes})
        self.out = nn.ModuleDict({t: nn.Linear(hidden, hidden, bias=False) for t in self.ntypes})
        self.wa, self.wm = nn.ParameterDict(), nn.ParameterDict()
        for r in self.etypes:
            a = torch.empty(heads, self.dk, self.dk); b = torch.empty(heads, self.dk, self.dk)
            nn.init.xavier_uniform_(a); nn.init.xavier_uniform_(b)
            self.wa[_rk(r)] = nn.Parameter(a); self.wm[_rk(r)] = nn.Parameter(b)

    def forward(self, x, ei):
        H, dk = self.H, self.dk
        agg = {t: x[t].new_zeros(x[t].size(0), H, dk) for t in x}
        for r, e in ei.items():
            st, _, dt = r; si, di = e[0], e[1]
            s = self.s[st](x[st]).view(-1, H, dk)[si]
            d = self.d[dt](x[dt]).view(-1, H, dk)[di]
            m = self.m[st](x[st]).view(-1, H, dk)[si]
            att = torch.einsum("ehi,hij,ehj->eh", s, self.wa[_rk(r)], d) / (dk ** 0.5)
            alpha = torch.softmax(att, dim=1)                 # across heads of each edge
            msg = torch.einsum("ehi,hij->ehj", m, self.wm[_rk(r)])
            agg[dt].index_add_(0, di, alpha.unsqueeze(-1) * msg)   # uniform neighbour sum
        return {t: self.out[t](agg[t].reshape(agg[t].size(0), H*dk)) + x[t] for t in x}

In [ ]:
class HGAE(nn.Module):
    def __init__(self, metadata, in_dims, tt, hidden=64, heads=16, layers=2, latent=64, dec=64, dropout=0.3):
        super().__init__()
        self.ntypes = metadata[0]; self.tt = tt; self.dropout = dropout
        self.inl = nn.ModuleDict({t: nn.Linear(in_dims[t], hidden) for t in self.ntypes})
        self.convs = nn.ModuleList([HGAEConv(metadata, hidden, heads) for _ in range(layers)])
        self.mu = nn.ModuleDict({t: nn.Linear(hidden, latent) for t in self.ntypes})
        self.lv = nn.ModuleDict({t: nn.Linear(hidden, latent) for t in self.ntypes})
        self.trunk, self.dcont, self.dcat = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
        for t in self.ntypes:
            self.trunk[t] = nn.Sequential(nn.Linear(latent, dec), nn.ReLU(), nn.Dropout(dropout),
                                          nn.Linear(dec, dec), nn.ReLU(), nn.Dropout(dropout))
            self.dcont[t] = nn.Linear(dec, max(1, tt[t]["cont"]))
            self.dcat[t] = nn.ModuleDict({n: nn.Linear(dec, w) for (n, _s, w) in tt[t]["cat_groups"]})

    def reparam(self, mu, lv):
        return mu + torch.randn_like(lv) * torch.exp(0.5*lv) if self.training else mu

    def encode(self, x, ei):
        h = {t: F.relu(self.inl[t](v)) for t, v in x.items()}
        mu = lv = None
        for c in self.convs:
            h = c(h, ei)
            h = {t: F.dropout(F.relu(v), p=self.dropout, training=self.training) for t, v in h.items()}
            mu = {t: self.mu[t](h[t]) for t in h}; lv = {t: self.lv[t](h[t]) for t in h}
            h = {t: self.reparam(mu[t], lv[t]) for t in h}
        return h, mu, lv

    def forward(self, x, ei):
        z, mu, lv = self.encode(x, ei)
        recon = {}
        for t in z:
            tr = self.trunk[t](z[t])
            recon[t] = {"cont": self.dcont[t](tr), "cats": {n: hd(tr) for n, hd in self.dcat[t].items()}}
        return recon, mu, lv

def ae_loss(recon, x, tt, prev, mu=None, lv=None, beta=0.0):
    total, parts = 0.0, {}
    for t, tg in tt.items():
        nc = tg["cont"]; l = F.mse_loss(recon[t]["cont"], x[t][:, :nc])
        for (n, s, w) in tg["cat_groups"]:
            l = l + F.cross_entropy(recon[t]["cats"][n], x[t][:, s:s+w].argmax(1))
        parts[t] = float(l.detach()); total = total + prev[t]*l
    if beta and mu is not None:
        kl = 0.0
        for t in mu:
            kl = kl + (-0.5*torch.mean(torch.sum(1 + lv[t] - mu[t].pow(2) - lv[t].exp(), dim=1)))
        total = total + beta*kl
    return total, parts

@torch.no_grad()
def txn_scores(model, data, tt, device):
    model.eval(); data = data.to(device)
    recon, _, _ = model(data.x_dict, data.edge_index_dict)
    x = data["transaction"].x; nc = tt["transaction"]["cont"]
    err = ((recon["transaction"]["cont"] - x[:, :nc])**2).mean(1)
    for (n, s, w) in tt["transaction"]["cat_groups"]:
        err = err + F.cross_entropy(recon["transaction"]["cats"][n], x[:, s:s+w].argmax(1), reduction="none")
    return err.cpu().numpy()

## 7. Training  (genuine-only, with per-epoch diagnostics)

In [ ]:
from sklearn.metrics import average_precision_score

model = HGAE(METADATA, IN_DIMS, TT, CFG["hidden_dim"], CFG["heads"], CFG["encoder_layers"],
             CFG["latent_dim"], CFG["decoder_hidden"], CFG["dropout"]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

counts = {t: train_data[t].x.size(0) for t in IN_DIMS}; tot = sum(counts.values())
prev = {t: counts[t]/tot for t in counts}                # Eq. 7 prevalence weights

hist = {"train_loss": [], "val_auc_pr": [], "err_genuine": [], "err_fraud": []}
best_auc, best_state = -1.0, None
for ep in range(1, CFG["epochs"]+1):
    model.train(); opt.zero_grad()
    recon, mu, lv = model(train_data.x_dict, train_data.edge_index_dict)
    loss, _ = ae_loss(recon, train_data.x_dict, TT, prev, mu, lv, CFG["beta_kl"])
    loss.backward(); opt.step()

    vs = txn_scores(model, val_data, TT, DEVICE)
    auc = float(average_precision_score(val_labels, vs))
    hist["train_loss"].append(float(loss.detach())); hist["val_auc_pr"].append(auc)
    hist["err_genuine"].append(float(vs[val_labels==0].mean()))
    hist["err_fraud"].append(float(vs[val_labels==1].mean()))
    if auc > best_auc + 1e-4:
        best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if ep % 5 == 0 or ep == 1:
        print(f"epoch {ep:3d}  train_loss={loss.item():.4f}  val_AUC-PR={auc:.4f}")

if best_state: model.load_state_dict(best_state)
print(f"done. best val AUC-PR = {best_auc:.4f}")

## 8. Validation &amp; thresholds, test metrics

In [ ]:
from sklearn.metrics import (precision_recall_curve, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score, confusion_matrix)

val_scores = txn_scores(model, val_data, TT, DEVICE)
gen_val = val_scores[val_labels == 0]
thr_mu2sigma = float(gen_val.mean() + 2*gen_val.std())            # paper Eq. 9 (genuine only)
p, r, thr = precision_recall_curve(val_labels, val_scores)
f1s = 2*p*r/(p+r+1e-12); thr_f1 = float(thr[np.nanargmax(f1s[:-1])])

test_scores = txn_scores(model, test_data, TT, DEVICE)
def report(name, T):
    pred = (test_scores >= T).astype(int)
    print(f"{name:12} thr={T:.4f}  P={precision_score(test_labels,pred,zero_division=0):.3f}  "
          f"R={recall_score(test_labels,pred,zero_division=0):.3f}  "
          f"F1={f1_score(test_labels,pred,zero_division=0):.3f}  "
          f"ROC-AUC={roc_auc_score(test_labels,test_scores):.3f}  "
          f"AUC-PR={average_precision_score(test_labels,test_scores):.3f}")
report("mu+2sigma", thr_mu2sigma)
report("F1-optimal", thr_f1)

## 9. Result visualizations

In [ ]:
ep = np.arange(1, len(hist["train_loss"])+1)
fig, ax = plt.subplots(2, 3, figsize=(16, 9))

# (a) training loss
ax[0,0].plot(ep, hist["train_loss"], color=ACCENT[0], lw=2)
ax[0,0].set_title("Training loss"); ax[0,0].set_xlabel("epoch"); ax[0,0].set_ylabel("loss")

# (b) validation AUC-PR
ax[0,1].plot(ep, hist["val_auc_pr"], color=ACCENT[2], lw=2)
ax[0,1].set_title("Validation AUC-PR"); ax[0,1].set_xlabel("epoch"); ax[0,1].set_ylabel("AUC-PR")

# (c) genuine vs fraud mean reconstruction error over epochs (the separation)
ax[0,2].plot(ep, hist["err_genuine"], color=GENUINE, lw=2, label="genuine")
ax[0,2].plot(ep, hist["err_fraud"], color=FRAUD, lw=2, label="fraud")
ax[0,2].set_title("Mean reconstruction error (val)"); ax[0,2].set_xlabel("epoch")
ax[0,2].set_ylabel("error"); ax[0,2].legend(frameon=False)

# (d) score distribution at the end, genuine vs fraud, with thresholds
b = np.linspace(test_scores.min(), np.percentile(test_scores, 99), 60)
ax[1,0].hist(test_scores[test_labels==0], bins=b, color=GENUINE, alpha=0.65, label="genuine", density=True)
ax[1,0].hist(test_scores[test_labels==1], bins=b, color=FRAUD, alpha=0.65, label="fraud", density=True)
ax[1,0].axvline(thr_mu2sigma, color=THRC, ls="--", lw=1.5, label="mu+2sigma")
ax[1,0].axvline(thr_f1, color=THRC, ls=":", lw=1.5, label="F1-optimal")
ax[1,0].set_title("Reconstruction-error distribution (test)"); ax[1,0].set_xlabel("error")
ax[1,0].set_ylabel("density"); ax[1,0].legend(frameon=False, fontsize=8)

# (e) ROC curve
fpr, tpr, _ = roc_curve(test_labels, test_scores)
ax[1,1].plot(fpr, tpr, color=ACCENT[0], lw=2, label=f"AUC={roc_auc_score(test_labels,test_scores):.3f}")
ax[1,1].plot([0,1],[0,1], color="#aaaaaa", ls="--", lw=1)
ax[1,1].set_title("ROC curve"); ax[1,1].set_xlabel("FPR"); ax[1,1].set_ylabel("TPR"); ax[1,1].legend(frameon=False)

# (f) Precision-Recall curve
pr_p, pr_r, _ = precision_recall_curve(test_labels, test_scores)
ax[1,2].plot(pr_r, pr_p, color=ACCENT[1], lw=2, label=f"AUC-PR={average_precision_score(test_labels,test_scores):.3f}")
ax[1,2].axhline(test_labels.mean(), color="#aaaaaa", ls="--", lw=1, label=f"base rate={test_labels.mean():.3f}")
ax[1,2].set_title("Precision-Recall curve"); ax[1,2].set_xlabel("recall"); ax[1,2].set_ylabel("precision"); ax[1,2].legend(frameon=False)

plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix at the F1-optimal threshold
pred = (test_scores >= thr_f1).astype(int)
cm = confusion_matrix(test_labels, pred)
fig, axc = plt.subplots(figsize=(4.5, 4))
im = axc.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        axc.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                 color="white" if cm[i,j] > cm.max()/2 else "#222222", fontweight="bold")
axc.set_xticks([0,1]); axc.set_xticklabels(["genuine","fraud"])
axc.set_yticks([0,1]); axc.set_yticklabels(["genuine","fraud"])
axc.set_xlabel("predicted"); axc.set_ylabel("actual")
axc.set_title(f"Confusion matrix @ F1 threshold ({thr_f1:.3f})"); axc.grid(False)
plt.tight_layout(); plt.show()

## 10. Score a single transaction

In [ ]:
# Build a 3-node graph (transaction + its customer + its merchant) and score it.
@torch.no_grad()
def score_one(row):
    xt = feats["transaction"].transform(pd.DataFrame([row]))
    xc = feats["customer"].transform(pd.DataFrame([row]))
    xm = feats["merchant"].transform(pd.DataFrame([row]))
    d = HeteroData()
    d["transaction"].x = torch.tensor(xt, dtype=torch.float32)
    d["customer"].x = torch.tensor(xc, dtype=torch.float32)
    d["merchant"].x = torch.tensor(xm, dtype=torch.float32)
    e = torch.tensor([[0],[0]])
    for r in METADATA[1]: d[r].edge_index = e.clone()
    err = float(txn_scores(model, d, TT, DEVICE)[0])
    return err, ("FRAUD" if err >= thr_f1 else "NON-FRAUD")

rng2 = np.random.default_rng(0)
def _sample(df, n):
    idx = rng2.choice(df.index.values, size=min(n, len(df)), replace=False)
    return df.loc[idx]

rows, correct = [], 0
for lbl, sub in [("genuine", _sample(test_raw[test_raw.is_fraud==0], 8)),
                 ("fraud",   _sample(test_raw[test_raw.is_fraud==1], 8))]:
    want = "FRAUD" if lbl == "fraud" else "NON-FRAUD"
    for _, row in sub.iterrows():
        err, verdict = score_one(row.to_dict())
        ok = (verdict == want); correct += ok
        rows.append({"actual": lbl, "error": round(err, 3), "verdict": verdict, "correct": ok})

print(f"Correct: {correct}/{len(rows)} at the F1 threshold ({thr_f1:.3f})")
pd.DataFrame(rows)   # rendered as a table in Colab